In [ ]:
#@title Konfiguracja środowiska
# Komórka 1: Konfiguracja Środowiska i Instalacja Zależności

import os

# Krok 1: Klonowanie repozytorium z modułami
print(" cloning repository...")
!git clone https://github.com/MattyMroz/ColabUniversalDownloader.git

# Przejście do katalogu repozytorium
os.chdir('ColabUniversalDownloader')
print(f" Zmieniono katalog roboczy na: {os.getcwd()}")

# Krok 2: Instalacja zależności z pliku requirements.txt
print("\n🐍 Instalowanie bibliotek Pythona z requirements.txt...")
!pip install -q -r requirements.txt
print("✅ Biblioteki Pythona zainstalowane pomyślnie.")

print("\n\n🎉 Środowisko jest gotowe do testów!")

In [ ]:
#@title Ustawienia (Form)

# Ścieżki katalogów tymczasowych
TMP_PIXELDRAIN = "./tmp_pixeldrain"  #@param {type:"string"}
TMP_MEGA = "./tmp_mega"  #@param {type:"string"}
TMP_YOUTUBE = "./tmp_youtube"  #@param {type:"string"}

# Parametry testów
DELETE_DELAY_SECONDS = 60  #@param {type:"number"}
CLEANUP_TMP_AFTER_TESTS = True  #@param {type:"boolean"}

# Linki testowe (możesz zmienić)
PIXEL_URL = "https://pixeldrain.com/u/e75isJ7y"  #@param {type:"string"}
MEGA_FILE_URL = "https://mega.nz/file/CVQXjbCZ#KhWlLh7Ec3z0EjcPqVX1_3ZyGQC05xNMs8gb_VYGdxg"  #@param {type:"string"}
MEGA_FOLDER_URL = "https://mega.nz/folder/SVxnDbbI#V5nwTs9FAXNlGzUmWBucIw"  #@param {type:"string"}
YOUTUBE_URL = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"  #@param {type:"string"}

# Docelowy dysk i folder w Google Drive
TARGET_DRIVE = "my"  #@param ["my", "shared"]
SHARED_DRIVE_NAME = ""  #@param {type:"string"}
SESSION_FOLDER_NAME = "ColabUniversalDownloader"  #@param {type:"string"}

# YouTube – globalne przełączniki (pod UI)
YTDL_VIDEO = True  #@param {type:"boolean"}
YTDL_AUDIO = True  #@param {type:"boolean"}
YTDL_SUBTITLES = False  #@param {type:"boolean"}
YTDL_SUB_LANGS = "pl,en"  #@param {type:"string"}
YTDL_BEST_MUXED = False  #@param {type:"boolean"}
YTDL_MAX_HEIGHT = 0  #@param {type:"number"}
YTDL_MAX_FPS = 0  #@param {type:"number"}
YTDL_VCODEC = ""  #@param {type:"string"}
YTDL_ACODEC = ""  #@param {type:"string"}
YTDL_CONCURRENT = True  #@param {type:"boolean"}
YTDL_PLAYLIST_ITEMS = ""  #@param {type:"string"}

In [ ]:
#@title Importy i inicjalizacja
# Komórka 2: Import Modułów i Inicjalizacja Narzędzi

from utils.google_drive import GoogleDriveManager
from utils.mega import MegaDownloader, make_refreshing_progress
from utils.pixeldrain import PixelDrainDownloader
from utils.youtube import YouTubeDownloader
import sys
from pprint import pprint

# Dodajemy ścieżkę do repozytorium, aby Python widział nasze moduły w katalogu 'utils'
sys.path.append('/content/ColabUniversalDownloader')


# --- Inicjalizacja klas ---
pixeldrain_handler = PixelDrainDownloader()
mega_handler = MegaDownloader()
yt_handler = YouTubeDownloader()
gdrive_handler = GoogleDriveManager()

print("✅ Moduły zaimportowane, obiekty gotowe do pracy.")
print(f"Google Drive Manager gotowy: {gdrive_handler.is_ready()}")

# --- Prosta funkcja do wyświetlania postępu ---


def simple_progress_callback(log_message):
    """Wyświetla każdą nową linię logu."""
    print(log_message)


# --- Callback do animowanego paska postępu (nadpisuje jedną linię) ---
import sys as _sys
from IPython.display import clear_output

# Szerokość paska postępu (znaki). Dostosuj w razie potrzeby.
PROGRESS_WIDTH = 100

def overwrite_line_progress(line: str, width: int = PROGRESS_WIDTH):
    """Drukuje ciągły postęp w jednej linii o stałej szerokości.
    - Przycina od lewej, aby zachować najważniejszy koniec (procenty/transfer)
    - Dopełnia spacjami, by nie "skakało"
    """
    if line is None:
        return
    msg = str(line).strip("\r\n")
    if not msg:
        return
    # Uporządkuj znaki sterujące
    msg = msg.replace("\r", " ").replace("\t", " ")
    # Przytnij od lewej, aby końcówka z postępem była widoczna
    if len(msg) > width:
        msg = "…" + msg[-(width - 1):]
    # Dopełnij, by przykryć ewentualne poprzednie dłuższe linie
    if len(msg) < width:
        msg = msg.ljust(width)
    try:
        _sys.stdout.write("\r" + msg)
        _sys.stdout.flush()
    except Exception:
        # Fallback: zwykły print
        print(msg)


class RefreshingProgress:
    """Odświeżany pasek postępu z headerem używający clear_output.
    Każda aktualizacja czyści wyjście komórki i drukuje nagłówek + linię postępu
    o stałej szerokości, by uniknąć „skakania”.
    """
    def __init__(self, header, width: int = PROGRESS_WIDTH):
        self.header_lines = header if isinstance(header, (list, tuple)) else [str(header)]
        self.width = width

    def __call__(self, line: str):
        if not line:
            return
        msg = str(line).strip("\r\n").replace("\r", " ").replace("\t", " ")
        # Przytnij od lewej i dopełnij do stałej szerokości
        if len(msg) > self.width:
            msg = "…" + msg[-(self.width - 1):]
        if len(msg) < self.width:
            msg = msg.ljust(self.width)
        clear_output(wait=True)
        for h in self.header_lines:
            print(h)
        print(msg)

In [ ]:
#@title Helpery (unikalne nazwy, filtrowanie logów, bez nadpisywania)
import os, re, time, shutil, uuid
from pathlib import Path

# Unikalny znacznik sesji, aby nie nadpisywać
SESSION_ID = time.strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:6]


def ensure_unique_dir(base_dir: str, label: str) -> str:
    """Zwraca ścieżkę do unikalnego podkatalogu w base_dir: base_dir/label-<SESSION_ID>."""
    p = Path(base_dir) / f"{label}-{SESSION_ID}"
    p.mkdir(parents=True, exist_ok=True)
    return str(p)


def safe_join(base_dir: str, filename: str) -> str:
    """Łączy katalog i nazwę pliku, nie pozwalając na wyjście ponad base_dir."""
    base = Path(base_dir).resolve()
    full = (base / filename).resolve()
    if base not in full.parents and base != full:
        # Jeśli ścieżka wychodzi poza katalog bazowy, przerzuć do bazowego
        full = base / Path(filename).name
    return str(full)


# Mega: filtr komunikatów do najważniejszych (postęp + podsumowania)
IMPORTANT_PATTERNS = [
    r"^\s*\d+%",             # linie postępu
    r"Downloaded ",           # ukończenia pobrań
    r"Downloading ",          # start pobierania
    r"^Error|^BŁĄD|^❌|^⚠️",  # błędy/ostrzeżenia
]
IMPORTANT_REGEX = re.compile("|".join(IMPORTANT_PATTERNS), re.IGNORECASE)

def mega_progress_minimal(line: str):
    if IMPORTANT_REGEX.search(line):
        print(line, end="")


# Pixeldrain: zbieraj do bufora i drukuj raz po zakończeniu
class SingleLineLogger:
    def __init__(self):
        self._buf = []
    def __call__(self, line: str):
        # ogranicz objętość bufora
        if len(self._buf) < 2000:
            self._buf.append(line.strip())
    def flush(self, header: str = None):
        if header:
            print(header)
        if self._buf:
            # pokaż tylko ostatnią linię lub streszczenie
            last = self._buf[-1]
            print(last)
        self._buf.clear()

single_line_logger = SingleLineLogger()


In [ ]:
#@title Test 1: Pixeldrain
# Komórka 3: Test Modułu `pixeldrain.py` — z użyciem katalogu tymczasowego

import os
import shutil

print("="*50)
print("🚀 TEST 1: POBIERANIE Z PIXELDRAIN 🚀")
print("="*50)

# Katalog tymczasowy (unikalny podkatalog sesji)
PXD_TMP_BASE = TMP_PIXELDRAIN
PXD_TMP = ensure_unique_dir(PXD_TMP_BASE, "pixeldrain")
os.makedirs(PXD_TMP, exist_ok=True)

filepath = None
prev_cwd = os.getcwd()

# Zbuduj header dla odświeżanego paska
header = [
    "==================== ZADANIE PIXELDRAIN ====================",
    f"URL: {PIXEL_URL}",
]
prog = RefreshingProgress(header)

try:
    # Pobieraj do katalogu tymczasowego (wget zapisze w aktualnym katalogu procesu)
    os.chdir(PXD_TMP)
    # Stabilny pasek postępu z clear_output
    filepath = pixeldrain_handler.download(PIXEL_URL, prog)
    # Po zakończeniu pokaż finalny ekran bez paska
    from IPython.display import clear_output as _clr
    _clr(wait=True)
    for h in header:
        print(h)

    if filepath and os.path.exists(filepath):
        print(f"\n✅ ZAKOŃCZONO POBIERANIE PIXELDRAIN")
        print(f"Plik: {filepath}")
        try:
            print(f"Rozmiar: {os.path.getsize(filepath) / 1024:.2f} KB")
        except Exception:
            pass
    else:
        print("\n❌ POBIERANIE PIXELDRAIN NIEUDANE")

finally:
    # Przywróć katalog roboczy i posprzątaj katalog tymczasowy (jeśli włączone)
    try:
        os.chdir(prev_cwd)
    except Exception:
        pass
    if CLEANUP_TMP_AFTER_TESTS:
        shutil.rmtree(PXD_TMP, ignore_errors=True)


In [ ]:
#@title Test 2: Mega.nz
# Komórka 4: Test Modułu `mega.py` — pobieranie pliku i całego folderu (bez listowania)

import os
import shutil
import re

print("="*50)
print("🚀 TEST 2: MEGA.NZ — PLIK I FOLDER 🚀")
print("="*50)

# Jeden wspólny katalog tymczasowy (unikalny podkatalog sesji)
MEGA_TMP_BASE = TMP_MEGA
MEGA_TMP = ensure_unique_dir(MEGA_TMP_BASE, "mega")
os.makedirs(MEGA_TMP, exist_ok=True)

# Header do odświeżanego progresu (spójny styl jak w Pixeldrain)
header_file = [
    "==================== ZADANIE MEGA — PLIK ====================",
    f"URL: {MEGA_FILE_URL}",
    "Nazwa pliku: —",
    "Rozmiar: —",
    "--------------------------------------------------",
]
header_folder = [
    "==================== ZADANIE MEGA — FOLDER ====================",
    f"URL: {MEGA_FOLDER_URL}",
    "Nazwa pliku: —",
    "Rozmiar: —",
    "--------------------------------------------------",
]

# --- Test A: Pobieranie pojedynczego pliku ---
prog_file = make_refreshing_progress(header_file)
print("\n--- TEST 2A: Pobieranie pliku ---")
file_path = mega_handler.download_file(
    MEGA_FILE_URL, dest_dir=MEGA_TMP, progress=prog_file)
from IPython.display import clear_output as _clr
_clr(wait=True)
for h in header_file:
    print(h)
if file_path and os.path.exists(file_path):
    print(f"\n✅ PLIK POBRANY: {file_path}")
    try:
        size_mb = os.path.getsize(file_path) / (1024*1024)
        print(f"   Rozmiar: {size_mb:.2f} MB")
    except Exception:
        pass
else:
    print("\n❌ Nie udało się pobrać pliku (sprawdź URL).")

# --- Test B: Pobieranie całego folderu ---
prog_folder = make_refreshing_progress(header_folder)
print("\n--- TEST 2B: Pobieranie folderu (całość) ---")
results = mega_handler.download_folder(
    MEGA_FOLDER_URL, dest_dir=MEGA_TMP, choose_files=False, progress=prog_folder)
_clr(wait=True)
for h in header_folder:
    print(h)

if results:
    print("\n✅ Pobrane pliki (wykryte w logu):")
    for p in results:
        print(f" - {p}")
else:
    # Fallback: wypisz cokolwiek pobrało się do folderu docelowego
    collected = []
    for root, _, files in os.walk(MEGA_TMP):
        for f in files:
            collected.append(os.path.join(root, f))
    if collected:
        print("\nℹ️ Nie wykryto plików w logu, ale znaleziono w systemie:")
        for p in collected:
            print(f" - {p}")
    else:
        print("\n❌ Nie udało się pobrać folderu (sprawdź URL).")

print("\n🎯 Zakończono testy Mega.")

# Sprzątanie pobranych plików po teście (opcjonalnie)
if CLEANUP_TMP_AFTER_TESTS:
    shutil.rmtree(MEGA_TMP, ignore_errors=True)

In [ ]:
#@title Test 3: Google Drive (upload Mega + Pixeldrain i linki)
# Komórka 5: Test Modułu `google_drive.py`

print("="*50)
print("🚀 TEST 3: OPERACJE NA GOOGLE DRIVE 🚀")
print("="*50)
print("UWAGA: Ta komórka wywoła okno autoryzacji Google. Zezwól na dostęp.")

if not gdrive_handler.is_ready():
    print("\n❌ TEST NIEUDANY. Manager Dysku Google nie został poprawnie zainicjalizowany.")
else:
    # 1) Ustal dysk docelowy i ID nadrzędnego miejsca
    is_shared = (TARGET_DRIVE == "shared")
    drive_name = SHARED_DRIVE_NAME if is_shared else "Mój Dysk"
    parent_id = gdrive_handler.get_drive_id(drive_name=drive_name, is_shared=is_shared)
    if not parent_id:
        print("❌ Nie udało się uzyskać ID dysku docelowego. Sprawdź nazwę dysku współdzielonego.")
    else:
        print(f"✅ Docelowy dysk: {'Współdzielony' if is_shared else 'Mój Dysk'} ({drive_name})")

        # 2) Upewnij się, że TMP katalogi istnieją (z unikalnymi podkatalogami sesyjnymi)
        from pathlib import Path
        MEGA_TMP_BASE = TMP_MEGA
        PXD_TMP_BASE = TMP_PIXELDRAIN
        MEGA_TMP = ensure_unique_dir(MEGA_TMP_BASE, "mega")
        PXD_TMP = ensure_unique_dir(PXD_TMP_BASE, "pixeldrain")
        Path(MEGA_TMP).mkdir(parents=True, exist_ok=True)
        Path(PXD_TMP).mkdir(parents=True, exist_ok=True)

        # 3) Pobierz ponownie pliki w tej komórce (z widocznym postępem)
        from IPython.display import clear_output as _clr
        print("\n➡️ Pobieranie z Mega (plik i folder)...")
        # Nagłówki i progres jak w Teście 2
        header_file = [
            "==================== ZADANIE MEGA — PLIK ====================",
            f"URL: {MEGA_FILE_URL}",
            "Nazwa pliku: —",
            "Rozmiar: —",
            "--------------------------------------------------",
        ]
        header_folder = [
            "==================== ZADANIE MEGA — FOLDER ====================",
            f"URL: {MEGA_FOLDER_URL}",
            "Nazwa pliku: —",
            "Rozmiar: —",
            "--------------------------------------------------",
        ]
        prog_file = make_refreshing_progress(header_file)
        prog_folder = make_refreshing_progress(header_folder)

        mega_file_path = mega_handler.download_file(
            MEGA_FILE_URL, dest_dir=MEGA_TMP, progress=prog_file)
        _clr(wait=True)
        for h in header_file:
            print(h)

        mega_folder_results = mega_handler.download_folder(
            MEGA_FOLDER_URL, dest_dir=MEGA_TMP, choose_files=False, progress=prog_folder)
        _clr(wait=True)
        for h in header_folder:
            print(h)

        print("\n➡️ Pobieranie z Pixeldrain...")
        import os as _os
        prev = _os.getcwd()
        _os.chdir(PXD_TMP)
        pix_file = pixeldrain_handler.download(PIXEL_URL, overwrite_line_progress)
        print()  # zakończ linię progresu
        _os.chdir(prev)

        # 4) Lista plików do wysłania (wszystko z katalogów tmp sesji) – deduplikacja po ścieżkach kanonicznych
        import os
        def collect_files(root_dir: str):
            files = []
            for r, _, fs in os.walk(root_dir):
                for f in fs:
                    files.append(os.path.join(r, f))
            return files

        files_to_upload = []
        files_to_upload += ([mega_file_path] if mega_file_path else [])
        files_to_upload += mega_folder_results
        files_to_upload += ([pix_file] if pix_file and os.path.exists(pix_file) else [])
        files_to_upload += collect_files(MEGA_TMP)
        files_to_upload += collect_files(PXD_TMP)
        # Deduplikacja: realpath + normcase (Windows/Unix)
        normed = []
        seen = set()
        for p in files_to_upload:
            if not p:
                continue
            rp = os.path.normcase(os.path.realpath(p))
            if rp in seen:
                continue
            seen.add(rp)
            normed.append(p)
        files_to_upload = normed
        print("\n📦 Pliki do wysłania (po deduplikacji):")
        for p in files_to_upload:
            print(f" - {p}")

        if not files_to_upload:
            print("\n❌ Nie znaleziono plików do wysłania.")
        else:
            # 5) Utwórz folder sesyjny ColabUniversalDownloader w wybranym dysku
            print("\n➡️ Tworzenie folderu sesyjnego na Dysku Google...")
            from googleapiclient.errors import HttpError
            try:
                folder_metadata = {
                    'name': SESSION_FOLDER_NAME,
                    'mimeType': 'application/vnd.google-apps.folder',
                    'parents': [parent_id]
                }
                folder = gdrive_handler.drive_service.files().create(
                    body=folder_metadata,
                    fields='id, webViewLink',
                    supportsAllDrives=True
                ).execute()
                session_folder_id = folder['id']
                session_folder_link = folder.get('webViewLink')
                print(f"✅ Folder utworzony: {SESSION_FOLDER_NAME} (ID: {session_folder_id})")
                print(f"🔗 Link do folderu: {session_folder_link}")
            except HttpError as he:
                print(f"⚠️ Folder już istnieje lub błąd: {he}")
                q = f"name = '{SESSION_FOLDER_NAME}' and mimeType = 'application/vnd.google-apps.folder' and '{parent_id}' in parents and trashed = false"
                resp = gdrive_handler.drive_service.files().list(q=q, fields='files(id, webViewLink)', supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
                items = resp.get('files', [])
                if items:
                    session_folder_id = items[0]['id']
                    session_folder_link = items[0].get('webViewLink')
                    print(f"ℹ️ Używam istniejącego folderu: {SESSION_FOLDER_NAME} (ID: {session_folder_id})")
                    print(f"🔗 Link do folderu: {session_folder_link}")
                else:
                    raise

            # 6) Upload wszystkich plików + linki publiczne
            print("\n➡️ Wysyłanie plików na Dysk Google i udostępnianie...")
            uploaded = []
            for p in files_to_upload:
                info = gdrive_handler.upload_and_share(p, session_folder_id, skip_if_exists=True, replace_if_exists=False)
                if info and info.get('id'):
                    if info.get('link'):
                        print(f"✅ Upload: {os.path.basename(p)}")
                        print(f"   🔗 Link: {info['link']}")
                    else:
                        print(f"✅ Upload: {os.path.basename(p)} (bez linku publicznego)")
                    uploaded.append(info)
                else:
                    print(f"❌ Nie udało się wysłać: {p}")

            # 7) Tylko kasowanie folderu sesyjnego (bez kasowania pojedynczych plików)
            delay = int(DELETE_DELAY_SECONDS)
            try:
                if delay <= 0:
                    print("\n🧹 Natychmiast usuwam folder sesyjny (po oczyszczeniu zawartości):")
                    print(f" - {session_folder_id}")
                    gdrive_handler.delete_folder_now(session_folder_id)
                else:
                    print(f"\n🧹 Zaplanowane usunięcie folderu: {session_folder_id} za {delay}s")
                    gdrive_handler.delete_folder_after_delay(session_folder_id, delay_seconds=delay)
            except Exception:
                pass

            # 8) Link do dysku i folderu
            print("\n📁 Podsumowanie:")
            print(f"Folder: {SESSION_FOLDER_NAME}")
            if session_folder_link:
                print(f"🔗 Przeglądaj folder: {session_folder_link}")
            else:
                base_drive = "https://drive.google.com/drive/"
                print(f"🔗 Dysk: {base_drive}{'shared-drives' if is_shared else 'my-drive'}")

        # 9) Sprzątanie katalogów tymczasowych po testach (jeśli włączone)
        if CLEANUP_TMP_AFTER_TESTS:
            import shutil as _sh
            _sh.rmtree(MEGA_TMP, ignore_errors=True)
            _sh.rmtree(PXD_TMP, ignore_errors=True)


In [ ]:
#@title Test 4: YouTube (listowanie + pobieranie)
# Komórka 6: Test Modułu `youtube.py`

print("="*50)
print("🚀 TEST 4: YOUTUBE — LISTOWANIE I POBIERANIE 🚀")
print("="*50)

import os

# TMP
YT_TMP_BASE = TMP_YOUTUBE
YT_TMP = ensure_unique_dir(YT_TMP_BASE, "youtube")
os.makedirs(YT_TMP, exist_ok=True)

# 1) Rozpoznanie linku i listowanie formatów
try:
    info = yt_handler.probe(YOUTUBE_URL)
    kind = info.get('_type') or 'video'
    title = info.get('title') or info.get('id') or 'Nieznany'
    print(f"🔎 Rozpoznano: {kind} — {title}")
    if kind == 'playlist':
        entries = info.get('entries') or []
        print(f"   Pozycji w playliście: {len(entries)}")
    
    print("\n📋 Dostępne formaty (top 5 w każdej grupie):")
    fmts = yt_handler.list_formats(YOUTUBE_URL)
    groups = {'muxed': [], 'video-only': [], 'audio-only': []}
    for f in fmts:
        groups.get(f['source'], groups['muxed']).append(f)
    for label in ['muxed', 'video-only', 'audio-only']:
        arr = groups.get(label, [])
        if not arr:
            continue
        print(f"\n— {label.upper()} —")
        for i, f in enumerate(arr[:5], 1):
            star = '⭐ ' if i == 1 else '  '
            h = f.get('height') or '-'
            fps = int(f.get('fps') or 0)
            size = f.get('filesize_mb') or 0
            print(f" {star}{i:2d}. {str(h)+'p':>5} {str(fps)+'fps':>6}  {f['vcodec']}/{f['acodec']}  {f['ext']}  ~{size:.2f} MB  id={f['format_id']}")
except Exception as e:
    print(f"❌ Błąd rozpoznania/listowania: {e}")

# 2) Pobieranie według ustawień formularza
print("\n➡️ Pobieranie z YouTube wg ustawień…")
header = [
    "==================== ZADANIE YOUTUBE ====================",
    f"URL: {YOUTUBE_URL}",
    "Nazwa: —",
    "Rozmiar: —",
    "--------------------------------------------------",
]
prog = RefreshingProgress(header)

# paramy
langs = [s.strip() for s in (YTDL_SUB_LANGS or '').split(',') if s.strip()]
langs = langs or ['pl','en']

outputs = []
try:
    outputs = yt_handler.download(
        YOUTUBE_URL,
        out_dir=YT_TMP,
        video=YTDL_VIDEO,
        audio=YTDL_AUDIO,
        subtitles=YTDL_SUBTITLES,
        subtitle_langs=langs,
        best_muxed=YTDL_BEST_MUXED,
        max_height=(YTDL_MAX_HEIGHT or None),
        max_fps=(YTDL_MAX_FPS or None),
        vcodec_preference=(YTDL_VCODEC or None),
        acodec_preference=(YTDL_ACODEC or None),
        progress=prog,
        concurrent=YTDL_CONCURRENT,
        playlist_items=(YTDL_PLAYLIST_ITEMS or None) if (YTDL_PLAYLIST_ITEMS or '').strip() else None,
    )
    from IPython.display import clear_output as _clr
    _clr(wait=True)
    for h in header:
        print(h)
    print("\n✅ Zakończono pobieranie. Pliki:")
    for p in outputs:
        print(f" - {p}")
except Exception as e:
    print(f"\n❌ Błąd pobierania: {e}")

# Sprzątanie TMP jeżeli włączone
if CLEANUP_TMP_AFTER_TESTS:
    import shutil as _sh
    _sh.rmtree(YT_TMP, ignore_errors=True)